# 完整拉回策略 × 富邦 API Pipeline

執行順序：
1. 富邦盤後成交、庫存、MFE、exit log、策略帳本更新
2. FinLab 收盤選股，排除富邦目前全部持股
3. 自動建立基本 order intent
4. 下一次盤後由富邦成交紀錄自動補入實際數量與成交價

本程式不會自動下單。


In [7]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\hh483\Desktop\FubonAPI\Complete_Pullback_Fubon_Pipeline_Package")
os.chdir(PROJECT_DIR)

for module_name in [
    "integrated_stock_pipeline_exitlog_fixed_strategy_ledger_v2",
    "integrated_stock_pipeline_strategy_complete_v2",
    "pullback_macdonly_daily_selector_v2",
    "complete_pullback_fubon_pipeline_v2",
]:
    sys.modules.pop(module_name, None)

from integrated_stock_pipeline_strategy_complete_v2 import PipelineConfig
from complete_pullback_fubon_pipeline_v2 import (
    CompletePipelineConfig,
    run_complete_daily_pipeline,
    run_selection_and_intent_only,
)

print("工作目錄：", Path.cwd())


工作目錄： C:\Users\hh483\Desktop\FubonAPI\Complete_Pullback_Fubon_Pipeline_Package


In [8]:
broker_config = PipelineConfig(
    target_date= None,
    google_credentials_file=Path("service_account.json"),
    spreadsheet_id="1-bs4-2mYutvQcUYY-np5zp-QqEJfab--TAjmfQi8RQY",
    sheet_name="市場廣度",
    enable_google_sheet=True,
    entry_condition_file=Path("holdings_entry_conditions.csv"),
    use_cache=False,
    output_dir=Path("output_integrated_stock"),
    cache_dir=Path("cache_integrated_stock"),
    strategy_ledger_dir=Path("strategy_ledger"),
    enable_strategy_ledger=True,
    lookback_days=365 * 1,
    filled_history_chunk_days=29,
    mfe_threshold=0.40,
    min_holding_trading_days=35,
    pullback_threshold=0.25,
    continue_on_market_error=True,
    continue_on_gsheet_error=True,
    continue_on_mfe_error=True,
    debug=True,
)

config = CompletePipelineConfig(
    project_dir=PROJECT_DIR,
    strategy_id="pullback_macd_day35_v1",
    trade_id_prefix="PB",
    intent_valid_calendar_days=5,
    selector_output_dir=Path("output_selector"),
    selector_signal_date=None,
    selector_total_equity=None,
    broker_config=broker_config,
    run_broker_first=True,
    create_order_intents=True,
)


## 每日盤後完整執行

正常情況每天只需執行以下一格。第一次會把既有庫存初始化為 `legacy`。


In [9]:
result = run_complete_daily_pipeline(config)


版本確認：complete_pullback_fubon_pipeline v1.0

========== A. 富邦盤後 / 策略帳本 ==========
版本確認：integrated_stock_pipeline v2.1-strategy-ledger
目標日期 target/as_of_date：2026-08-18
cache：每次更新

[STEP] 讀取入場條件清單

[STEP] 登入富邦 API

[STEP] 取得目前庫存（整股 + 零股）

[STEP] 取得成交紀錄


取得成交紀錄:   0%|                                                                             | 0/13 [00:00<?, ?it/s]

20250818 ~ 20250823 OK 0


取得成交紀錄:   8%|█████▎                                                               | 1/13 [00:02<00:24,  2.04s/it]

20250824 ~ 20250922 OK 0


...

20260620 ~ 20260719 OK 58


取得成交紀錄:  92%|██████████████████████████████████████████████████████████████▊     | 12/13 [00:24<00:02,  2.10s/it]

20260720 ~ 20260818 OK 14


取得成交紀錄: 100%|████████████████████████████████████████████████████████████████████| 13/13 [00:26<00:00,  2.08s/it]



[STEP] 整理目標日買入 / 賣出

[STEP] 建立市場廣度記錄持股 universe

[STEP] 依條件分類：突破 / 回檔 / 排除

[INFO] 目標日買入： []
[INFO] 目標日賣出： []
[INFO] 排除標的： []
[INFO] 廣度記錄持股（已排除）： []
[INFO] 入場條件分類： {'突破': 0, '回檔': 0, '排除': 0}

[STEP] 取得 TWSE/TPEX 市場廣度快照

[STEP] 寫入 Google Sheet
已更新 Google Sheet：2026-08-18

[STEP] FIFO 重建目前庫存批次（已排除）

[STEP] 彙總持股（holding days 使用 target_date，已排除）

[STEP] 計算 MFE / 回檔（價格抓到 target_date）

[STEP] 更新策略帳本 / 庫存核對

[STEP] 輸出 Excel / parquet

=== 出場觀察 ===
條件 A：MFE > 40% 且浮盈回吐 > 25 個百分點；條件 B：持有 >= 35 個交易日；任一成立即列入。
目前沒有符合出場觀察條件的股票，或 MFE 未成功產生。

輸出檔案：
excel: C:\Users\hh483\Desktop\FubonAPI\Complete_Pullback_Fubon_Pipeline_Package\output_integrated_stock\integrated_stock_report_20260818_20260818_231955.xlsx
mfe_parquet: C:\Users\hh483\Desktop\FubonAPI\Complete_Pullback_Fubon_Pipeline_Package\output_integrated_stock\integrated_mfe_all_20260818_20260818_231955.parquet
alert_parquet: C:\Users\hh483\Desktop\FubonAPI\Complete_Pullback_Fubon_Pipeline_Package\output_integrated_stock\integrated_exit_watch_2026

## 執行結果檢查


In [10]:
selection = result["selection"]
print("Signal date:", selection["signal_date"])
print("Candidates:", len(selection["all_candidates"]))
print("Buy list:", len(selection["buy_list"]))
print("New intents:", len(result["new_order_intents"]))
print("Intent path:", result["order_intent_path"])

display(selection["buy_list"])
display(result["new_order_intents"])


Signal date: 2026-08-18 00:00:00
Candidates: 0
Buy list: 0
New intents: 0
Intent path: C:\Users\hh483\Desktop\FubonAPI\Complete_Pullback_Fubon_Pipeline_Package\strategy_ledger\order_intent.parquet


,rank,signal_date,stock_id,bias_at_signal,macd_osc_at_signal,macd_osc_change_at_signal,foreign_ratio_at_signal,close_at_signal,market_drawdown,market_adx,market_plus_di,market_minus_di,already_held,suggested_position_value,selected_for_next_open


,trade_id,strategy_id,signal_date,expected_order_date,expires_date,stock_no,side,status,created_at,updated_at,order_no,filled_date,filled_quantity,filled_price,signal_rank,bias_at_signal,note


In [11]:
broker = result.get("broker")
if broker and broker.get("strategy_ledger"):
    ledger = broker["strategy_ledger"]
    recon = ledger["reconciliation"]
    review = recon[recon["status"].ne("OK")]
    print("策略 open lots：", len(ledger["lots"][ledger["lots"]["status"].eq("open")]))
    print("需要人工核對的庫存：", len(review))
    display(review)
    
    exec_review = ledger["execution_log"]
    exec_review = exec_review[exec_review["review_required"].fillna(False)]
    print("需要人工核對的成交：", len(exec_review))
    display(exec_review.tail(30))


策略 open lots： 0
需要人工核對的庫存： 0


,reconciliation_date,stock_no,total_today_qty,remaining_quantity,difference,status


需要人工核對的成交： 72


,execution_key,date,stock_no,side,qty,price,time,order_no,filled_no,trade_id,strategy_id,allocation_method,review_required,processed_at
550,filled:02000149675,2026-07-14,8936,sell,181,55.00,09:11:12.117,o9978,02000149675,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:31.989571
551,filled:02000149676,2026-07-14,8936,sell,50,55.00,09:11:12.117,o9978,02000149676,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:31.997632
552,filled:02000149677,2026-07-14,8936,sell,25,55.00,09:11:12.117,o9978,02000149677,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.008072
553,filled:02000149678,2026-07-14,8936,sell,31,55.00,09:11:12.117,o9978,02000149678,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.022276
554,filled:02000159330,2026-07-14,8227,sell,135,174.00,09:11:43.821,oA198,02000159330,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.034985
555,filled:02000762383,2026-07-14,6672,sell,5,274.00,09:12:22.038,oA444,02000762383,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.043065
556,filled:02000762384,2026-07-14,6672,sell,100,274.00,09:12:22.038,oA444,02000762384,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.055660
557,filled:02000762385,2026-07-14,6672,sell,25,274.00,09:12:22.038,oA444,02000762385,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.070248
558,filled:02000820779,2026-07-14,3661,sell,1,3735.00,09:13:01.430,oA709,02000820779,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.082364
559,filled:02000838068,2026-07-14,3661,sell,2,3740.00,09:13:16.527,oA805,02000838068,,unassigned,unmatched_fifo_all_strategies,True,2026-07-14 18:37:32.094468


## 只跑選股＋intent（不連富邦）

僅在同一天已經完成富邦盤後流程、需要重新執行選股時使用。它會讀取已保存的 strategy lots 作為持股清單。


In [12]:
# selection_only_result = run_selection_and_intent_only(config)
